# Easy Q1 — 台灣本島 2024 年每日降雨量資料庫

## 1. 資料集選擇說明

選用 **IMERG Final Run v06/v07（`NASA/GPM_L3/IMERG_V06`）**：

| 屬性 | 說明 |
|------|------|
| 時間解析度 | 30 分鐘（可聚合為每日） |
| 空間解析度 | 0.1° × 0.1°（約 11 km） |
| 單位 | mm/hr（需 × 0.5 × 24 轉為 mm/day） |
| 可用時間範圍 | 2000-06-01 至今（延遲約 3.5 個月） |
| 台灣適用性 | 全球覆蓋，包含台灣，優於 CHIRPS（CHIRPS 僅 50°S–50°N 陸地）；對山地降雨有衛星-雨量計合併校正 |

In [1]:
import ee
import xarray as xr
import numpy as np
import pandas as pd
from datetime import date

ee.Authenticate()
ee.Initialize(project='silicon-pattern-423512-i3')

In [2]:
# 台灣本島 AOI
taiwan_bbox = ee.Geometry.Rectangle([119.9, 21.8, 122.1, 25.4])

START = '2024-01-01'
END   = '2024-12-31'

# GPM IMERG 30-min precipitation rate (mm/hr)
imerg = (ee.ImageCollection('NASA/GPM_L3/IMERG_V06')
         .filterDate(START, END)
         .filterBounds(taiwan_bbox)
         .select('precipitationCal'))

print('影像數（30-min frames）:', imerg.size().getInfo())

/home/wugliang/mask_model/lib/python3.10/site-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for NASA/GPM_L3/IMERG_V06! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by NASA/GPM_L3/IMERG_V07

Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V06

  warnings.warn(warning, category=DeprecationWarning)


影像數（30-min frames）: 7381


In [3]:
# 聚合為每日降雨量（mm/day）
# precipitationCal 單位 mm/hr；每 30 min 影像代表 0.5 hr → mm/day = sum(value * 0.5)

def daily_sum(day_offset):
    day = ee.Date(START).advance(day_offset, 'day')
    next_day = day.advance(1, 'day')
    daily = (imerg
             .filterDate(day, next_day)
             .map(lambda img: img.multiply(0.5))  # mm/hr * 0.5 hr = mm
             .sum()
             .set('system:time_start', day.millis())
             .set('date', day.format('YYYY-MM-dd')))
    return daily

n_days = 366  # 2024 是閏年
daily_col = ee.ImageCollection(
    ee.List.sequence(0, n_days - 1).map(daily_sum)
)
print('每日影像數:', daily_col.size().getInfo())

每日影像數: 366


In [5]:
# 透過 ee.data.getPixels / xee 轉 xarray Dataset
# 使用 xee（Google 官方 Earth Engine xarray backend）
import xee

ds = xr.open_dataset(
    daily_col,
    engine='ee',
    projection=daily_col.first().projection(),
    geometry=taiwan_bbox,
    scale=11132,  # ~0.1 deg in meters at equator
)

# 重命名維度與變數
ds = ds.rename({'precipitationCal': 'precip_mm_day', 'lon': 'x', 'lat': 'y'})
print(ds)

TypeError: EarthEngineBackendEntrypoint.open_dataset() got an unexpected keyword argument 'projection'

In [6]:
# 加入 metadata
from datetime import datetime

ds.attrs.update({
    'dataset_id'          : 'NASA/GPM_L3/IMERG_V06',
    'variable_name'       : 'precipitationCal',
    'unit'                : 'mm/day',
    'spatial_resolution'  : '0.1 degree (~11 km)',
    'temporal_resolution' : 'daily (aggregated from 30-min)',
    'CRS'                 : 'EPSG:4326',
    'AOI'                 : 'Taiwan island [119.9, 21.8, 122.1, 25.4]',
    'processing_date'     : datetime.today().strftime('%Y-%m-%d'),
})

print('Metadata:', ds.attrs)

NameError: name 'ds' is not defined

In [ ]:
# ── 資料驗證 ────────────────────────────────────────────────────────────

times = pd.to_datetime(ds.time.values)

# 1. 時間連續性 & 缺日
expected = pd.date_range('2024-01-01', '2024-12-31', freq='D')
missing_days = expected.difference(times)
print(f'預期天數: {len(expected)}, 實際天數: {len(times)}')
print(f'缺失日期: {list(missing_days.strftime("%Y-%m-%d")) if len(missing_days) else "無"}')

# 2. 降雨量不合理負值
neg_count = int((ds['precip_mm_day'] < 0).sum())
print(f'負值像素數: {neg_count}')

# 3. 基本統計
print('\n基本統計：')
print(ds['precip_mm_day'].to_series().describe())

In [ ]:
# 輸出 Zarr（以月份為 time chunk，空間 chunk 32×32）
zarr_path = 'taiwan_precip_2024.zarr'

ds_chunked = ds.chunk({'time': 30, 'y': 32, 'x': 32})
ds_chunked.to_zarr(zarr_path, mode='w')
print(f'已輸出至 {zarr_path}')

In [ ]:
# 驗證讀取 Zarr
ds_check = xr.open_zarr(zarr_path)
print(ds_check)
print('Metadata:', ds_check.attrs)